## Executive Summary
Write a short (200 words max) summary of what the report contains and what you have discovered.

## Introduction
Write a short (200 words max) introduction to the dataset, emphasising why you chose it and what you expect to discover (what you expect to discover does not have to be what you then eventually discover)

## Data Cleaning

The dataset consists of 20 Project Gutenberg text files from a selection of popular science fiction books by women authors. The preparation process will check accessibility, inspect the text structure and identify material that should be excluded from analysis.

### Checking the Dataset

The first step counts the `.txt` files in the dataset to confirm that Python has found all 20 books. This helps to detect missing or incorrectly stored files before data cleaning begins. However, locating a file does not guarantee that it can be opened and imported successfully.

In [1]:
from pathlib import Path

book_folder = Path("Gutenberg_books")
book_files = sorted(book_folder.glob("*.txt"))

print("Number of books found:", len(book_files))

Number of books found: 20


#### Verifying File Accessibility

In [2]:
for book in book_files:
    try:
        with open(book, encoding="utf-8-sig") as file:
            content = file.read()

        print(book.name, "- Successfully opened")

    except (OSError, UnicodeError) as error:
        print(book.name, "- Error:", error)

1250-0.txt - Successfully opened
29019-0.txt - Successfully opened
84-0.txt - Successfully opened
pg16921.txt - Successfully opened
pg18247.txt - Successfully opened
pg18458.txt - Successfully opened
pg18846.txt - Successfully opened
pg19029.txt - Successfully opened
pg19090.txt - Successfully opened
pg19145.txt - Successfully opened
pg19651.txt - Successfully opened
pg19726.txt - Successfully opened
pg20788.txt - Successfully opened
pg20796.txt - Successfully opened
pg22544.txt - Successfully opened
pg23301.txt - Successfully opened
pg25550.txt - Successfully opened
pg29809.txt - Successfully opened
pg30427.txt - Successfully opened
pg32664.txt - Successfully opened


**Observation:** Python identified 20 `.txt` files, matching the expected number of works in the dataset. The subsequent accessibility check confirmed that all 20 files could be opened using the selected "utf-8-sig" encoding.

**Implication:** This establishes that the files are accessible for further processing, although it does not confirm that their contents are consistently formatted or ready for analysis.

### Inspecting a Sample File

The next step tests the import of one file in the dataset before continuing with the entire dataset, "84-0.txt" (*Frankenstein* by Mary Shelley).

In [3]:
book_path = Path("Gutenberg_books") / "84-0.txt"

with open(book_path, encoding="utf-8-sig") as file:
    text = file.read()

print("File loaded successfully!")

File loaded successfully!


To check for non-literary content, the first and last 1,000 characters of the imported text are displayed. This shows that the file contains additional material including publication dates, language information, character encoding and other metadata. Because this investigation compares literary writing styles, including sentence length and structure, this content must be identified and removed before analysis to avoid distorting the results.

In [4]:
print(text[:1000])

The Project Gutenberg eBook of Frankenstein, by Mary Wollstonecraft (Godwin) Shelley

This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online at
www.gutenberg.org. If you are not located in the United States, you
will have to check the laws of the country where you are located before
using this eBook.

Title: Frankenstein
       or, The Modern Prometheus

Author: Mary Wollstonecraft (Godwin) Shelley

Release Date: 31, 1993 [eBook #84]
[Most recently updated: November 13, 2020]

Language: English

Character set encoding: UTF-8

Produced by: Judith Boss, Christy Phillips, Lynn Hanninen, and David Meltzer. HTML version by Al Haines.
Further corrections by Menno de Leeuw.

*** START OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN ***




Frankenstein;

or, the Modern Prom

In [5]:
print(text[-1000:])

w.gutenberg.org/donate

Section 5. General Information About Project Gutenberg-tm electronic works

Professor Michael S. Hart was the originator of the Project
Gutenberg-tm concept of a library of electronic works that could be
freely shared with anyone. For forty years, he produced and
distributed Project Gutenberg-tm eBooks with only a loose network of
volunteer support.

Project Gutenberg-tm eBooks are often created from several printed
editions, all of which are confirmed as not protected by copyright in
the U.S. unless a copyright notice is included. Thus, we do not
necessarily keep eBooks in compliance with any particular paper
edition.

Most people start at our website which has the main PG search
facility: www.gutenberg.org

This website includes information about Project Gutenberg-tm,
including how to make donations to the Project Gutenberg Literary
Archive Foundation, how to help produce our new eBooks, and how to
subscribe to our email newsletter to hear about new eBooks.




**Observation:** The start and end of *Frankenstein* show that the raw text includes Project Gutenberg information as well as the novel. This includes publication details and licensing information, which is not part of Shelley’s writing.

**Implication:** Leaving this material in could affect vocabulary counts, sentence-length calculations and other style comparisons. The text therefore needs to be cleaned before further analysis.

### Locating the Gutenberg Markers

The next step is to identify the boundaries of the eBook content, starting with *Frankenstein*. The Gutenberg header, footer and licensing information are not part of the author's writing, and including them could distort comparisons in the analysis. The start and end markers will be located using Python's `.find()` method (Python Software Foundation, n.d.). Although these markers identify the boundaries of the eBook content, additional material may still be present within them.

In [6]:
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_position = text.find(start_marker)
end_position = text.find(end_marker)

print("Start marker position:", start_position)
print("End marker position:", end_position)

Start marker position: 904
End marker position: 420306


For the *Frankenstein* test case, the search found the Project Gutenberg start marker at character position 904 and the end marker at position 420,306. Both markers appeared in the expected order, which indicates that the file has clear boundaries between the novel and the surrounding Gutenberg text. These boundaries can be used to remove the header and footer, although the text between them still needs to be inspected for any remaining content which is not part of the literary work.

### Marker Variations Across the Dataset

Manual inspection of the wider dataset showed that both markers vary. The start of the novel is marked by either “`*** START OF THE PROJECT GUTENBERG EBOOK`” or “`*** START OF THIS PROJECT GUTENBERG EBOOK`”, and the end by either “`*** END OF THE PROJECT GUTENBERG EBOOK`” or “`*** END OF THIS PROJECT GUTENBERG EBOOK`”. The cleaning code must therefore check for both variations of each marker so that the boundaries are found consistently across all 20 files.

In [7]:
start_markers = [
    "*** START OF THIS PROJECT GUTENBERG EBOOK",
    "*** START OF THE PROJECT GUTENBERG EBOOK"
]

end_markers = [
    "*** END OF THIS PROJECT GUTENBERG EBOOK",
    "*** END OF THE PROJECT GUTENBERG EBOOK"
]

for book in book_files:
    with open(book, encoding="utf-8-sig") as file:
        content = file.read()

    start_positions = [content.find(marker) for marker in start_markers]
    end_positions = [content.find(marker) for marker in end_markers]

    print(book.name, "Start:", start_positions, "End:", end_positions)

1250-0.txt Start: [-1, 727] End: [-1, 108837]
29019-0.txt Start: [-1, 789] End: [-1, 17485]
84-0.txt Start: [-1, 904] End: [-1, 420306]
pg16921.txt Start: [447, -1] End: [337250, -1]
pg18247.txt Start: [-1, 686] End: [-1, 1000734]
pg18458.txt Start: [413, -1] End: [348761, -1]
pg18846.txt Start: [422, -1] End: [136135, -1]
pg19029.txt Start: [444, -1] End: [34151, -1]
pg19090.txt Start: [432, -1] End: [190230, -1]
pg19145.txt Start: [430, -1] End: [353370, -1]
pg19651.txt Start: [441, -1] End: [320060, -1]
pg19726.txt Start: [482, -1] End: [238516, -1]
pg20788.txt Start: [432, -1] End: [351576, -1]
pg20796.txt Start: [453, -1] End: [261522, -1]
pg22544.txt Start: [-1, -1] End: [-1, -1]
pg23301.txt Start: [429, -1] End: [36406, -1]
pg25550.txt Start: [443, -1] End: [321261, -1]
pg29809.txt Start: [512, -1] End: [457450, -1]
pg30427.txt Start: [468, -1] End: [122697, -1]
pg32664.txt Start: [439, -1] End: [132478, -1]


The initial search results identified two Gutenberg marker variations across 19 files, while one file did not match either variation. Each result contains two values corresponding to the marker variations in the order they were searched. For example, Start: [-1, 727] End: [-1, 108837] indicates that the first marker variation returned -1 (no match), while the second variation was found at character positions 727 and 108,837 for the start and end markers respectively. This allowed the matching variation to be identified for each file. The distribution is summarised below.


<table style="text-align:left;">
  <thead>
    <tr>
      <th>Marker variation</th>
      <th>Number of files</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>THIS PROJECT GUTENBERG EBOOK</td>
      <td>15</td>
    </tr>
    <tr>
      <td>THE PROJECT GUTENBERG EBOOK</td>
      <td>4</td>
    </tr>
    <tr>
      <td>Neither recognised</td>
      <td>1</td>
    </tr>
    <tr>
      <td><strong>Total</strong></td>
      <td><strong>20</strong></td>
    </tr>
  </tbody>
</table>


### Investigating Inconsistent Gutenberg Markers

The initial search successfully identified both start and end markers in 19 of the 20 files. However, `pg22544.txt` returned `[-1, -1]` for both searches, indicating that neither of the two specified marker variations was found.

Further manual inspection of this file, *A World Is Born*, revealed that its start marker uses a slightly different format:

`***START OF THE PROJECT GUTENBERG EBOOK A WORLD IS BORN***`

Unlike the previously identified variations, there is no space between the opening three asterisks and the word `START`. Since Python's `.find()` method searches for an exact sequence of characters, this spacing difference explains why the original search failed.

To address this inconsistency, a third start-marker variation has been added to the search criteria. A corresponding end-marker variation has also been included for testing. The amended code below repeats the search across all 20 files to determine whether both boundaries can now be identified consistently.

In [8]:

# Include the additional Gutenberg marker variations
start_markers = [
    "*** START OF THIS PROJECT GUTENBERG EBOOK",
    "*** START OF THE PROJECT GUTENBERG EBOOK",
    "***START OF THE PROJECT GUTENBERG EBOOK"
]

end_markers = [
    "*** END OF THIS PROJECT GUTENBERG EBOOK",
    "*** END OF THE PROJECT GUTENBERG EBOOK",
    "***END OF THE PROJECT GUTENBERG EBOOK"
]

# Search all 20 files for the start and end markers
for book in book_files:
    with open(book, encoding="utf-8-sig") as file:
        content = file.read()

    start_positions = [content.find(marker) for marker in start_markers]
    end_positions = [content.find(marker) for marker in end_markers]

    print(book.name, "Start:", start_positions, "End:", end_positions)


1250-0.txt Start: [-1, 727, -1] End: [-1, 108837, -1]
29019-0.txt Start: [-1, 789, -1] End: [-1, 17485, -1]
84-0.txt Start: [-1, 904, -1] End: [-1, 420306, -1]
pg16921.txt Start: [447, -1, -1] End: [337250, -1, -1]
pg18247.txt Start: [-1, 686, -1] End: [-1, 1000734, -1]
pg18458.txt Start: [413, -1, -1] End: [348761, -1, -1]
pg18846.txt Start: [422, -1, -1] End: [136135, -1, -1]
pg19029.txt Start: [444, -1, -1] End: [34151, -1, -1]
pg19090.txt Start: [432, -1, -1] End: [190230, -1, -1]
pg19145.txt Start: [430, -1, -1] End: [353370, -1, -1]
pg19651.txt Start: [441, -1, -1] End: [320060, -1, -1]
pg19726.txt Start: [482, -1, -1] End: [238516, -1, -1]
pg20788.txt Start: [432, -1, -1] End: [351576, -1, -1]
pg20796.txt Start: [453, -1, -1] End: [261522, -1, -1]
pg22544.txt Start: [-1, -1, 505] End: [-1, -1, 47993]
pg23301.txt Start: [429, -1, -1] End: [36406, -1, -1]
pg25550.txt Start: [443, -1, -1] End: [321261, -1, -1]
pg29809.txt Start: [512, -1, -1] End: [457450, -1, -1]
pg30427.txt Start

**Validation of the Revised Marker Search**

The amended search successfully identified both Gutenberg start and end markers in all 20 text files. In particular, `pg22544.txt`, which previously returned `[-1, -1]`, now returned `[-1, -1, 505]` for the start marker and `[-1, -1, 47993]` for the end marker. This confirms that the third variation, which omits the space after the opening asterisks, is used in both the header and footer of this file.

The final results identified three formatting variations across the dataset:

| Marker variation | Number of files |
|---|---:|
| `*** START OF THIS PROJECT GUTENBERG EBOOK` | 15 |
| `*** START OF THE PROJECT GUTENBERG EBOOK` | 4 |
| `***START OF THE PROJECT GUTENBERG EBOOK` | 1 |
| **Total** | **20** |

All 20 files now have identifiable start and end boundaries, allowing the cleaning process to proceed using a consistent set of search criteria. However, further inspection will still be required to identify any further material contained within these boundaries that is not part of the novel.

### Extracting the Gutenberg eBook Content

Having successfully identified the start and end markers in all 20 files, the next stage is to extract the text contained between these boundaries. This will remove the surrounding Project Gutenberg header and footer information, which is not part of the literary work and could affect subsequent text analysis.

The extraction process will initially be tested using *Frankenstein* before being applied to the complete dataset. The text will be extracted from the line following the start marker up to the beginning of the end marker, ensuring that the marker text itself is excluded.

Further inspection will be necessary to identify any remaining non-literary material, such as tables of contents, publication information or introductory notes.

In [9]:
# Load Frankenstein as the sample text
book_path = book_folder / "84-0.txt"

with open(book_path, encoding="utf-8-sig") as file:
    content = file.read()

# Find the matching start and end markers
start_position = next(
    content.find(marker)
    for marker in start_markers
    if marker in content
)

end_position = next(
    content.find(marker)
    for marker in end_markers
    if marker in content
)

# Move past the complete start-marker line
text_start = content.find("\n", start_position) + 1

# Extract the text and remove leading/trailing whitespace
extracted_text = content[text_start:end_position].strip()

print("Original characters:", len(content))
print("Extracted characters:", len(extracted_text))


Original characters: 438876
Extracted characters: 419335


In [10]:

# Display the first 1000 characters
print("BEGINNING OF FIRST 1000 CHARACTERS OF EXTRACTED TEXT:")
print("-" * 60)
print(extracted_text[:1000].lstrip())
print("-" * 60)
print("END OF FIRST 1000 CHARACTERS OF EXTRACTED TEXT")

# Add a gap between sections
print("\n" * 2)

# Display the last 1000 characters
print("BEGINNING OF LAST 1000 CHARACTERS OF EXTRACTED TEXT:")
print("-" * 60)
print(extracted_text[-1000:].rstrip())
print("-" * 60)
print("END OF LAST 1000 CHARACTERS OF EXTRACTED TEXT")


BEGINNING OF FIRST 1000 CHARACTERS OF EXTRACTED TEXT:
------------------------------------------------------------
Frankenstein;

or, the Modern Prometheus

by Mary Wollstonecraft (Godwin) Shelley


 CONTENTS

 Letter 1
 Letter 2
 Letter 3
 Letter 4
 Chapter 1
 Chapter 2
 Chapter 3
 Chapter 4
 Chapter 5
 Chapter 6
 Chapter 7
 Chapter 8
 Chapter 9
 Chapter 10
 Chapter 11
 Chapter 12
 Chapter 13
 Chapter 14
 Chapter 15
 Chapter 16
 Chapter 17
 Chapter 18
 Chapter 19
 Chapter 20
 Chapter 21
 Chapter 22
 Chapter 23
 Chapter 24




Letter 1

_To Mrs. Saville, England._


St. Petersburgh, Dec. 11th, 17—.


You will rejoice to hear that no disaster has accompanied the
commencement of an enterprise which you have regarded with such evil
forebodings. I arrived here yesterday, and my first task is to assure
my dear sister of my welfare and increasing confidence in the success
of my undertaking.

I am already far north of London, and as I walk in the streets of
Petersburgh, I feel a cold northern

***Observation: Remaining Non-Literary Content and Cleaning Limitations***

The first and last 1,000 characters of the extracted *Frankenstein* text suggest that the Project Gutenberg header and footer have been removed. However, the start of the text still contains the title, author information and a table of contents listing the letters and chapters before the narrative begins at "Letter 1".

Although the table of contents is part of the published eBook, it is not part of the novel itself. Retaining it would add extra words and distort sentence length and word frequency measurements.

Removing this material programmatically presents a further challenge because the 20 books may contain different types and arrangements of introductory content. Developing individual cleaning rules for each book would increase the complexity of the process and may introduce inconsistencies or accidentally remove genuine literary content.

For this analysis, the cleaning process will therefore prioritise the consistent removal of identifiable Project Gutenberg headers and footers across all 20 files. Any remaining front matter will be recognised as a limitation of the dataset and considered when interpreting the analytical results. Where practical, additional measures will be used during analysis to reduce the influence of non-narrative material.

### Applying Gutenberg Boundary Cleaning to the Complete Dataset

Having successfully identified the Gutenberg start and end markers in all 20 files, the extraction process can now be applied consistently across the complete dataset.

The code below uses the previously identified marker variations to locate the beginning and end of each eBook. It then extracts the text between these boundaries, excluding the Gutenberg marker lines and the surrounding administrative information.

The extracted texts will be stored in a Python dictionary, allowing each book to be accessed by its filename without modifying the original files.

A validation check will then confirm that all 20 books have been extracted successfully and that the resulting texts contain content. Remaining front matter, such as tables of contents and introductory information, will be acknowledged as a limitation during subsequent analysis.

In [11]:

# Store the extracted text for each book
extracted_books = {}

for book in book_files:
    with open(book, encoding="utf-8-sig") as file:
        content = file.read()

    # Find recognised Gutenberg marker positions
    start_positions = [
        content.find(marker) for marker in start_markers
    ]

    end_positions = [
        content.find(marker) for marker in end_markers
    ]

    # Exclude unsuccessful searches (-1)
    valid_starts = [pos for pos in start_positions if pos != -1]
    valid_ends = [pos for pos in end_positions if pos != -1]

    if valid_starts and valid_ends:
        start_position = min(valid_starts)
        end_position = min(valid_ends)

        # Locate the end of the start-marker line
        line_end = content.find("\n", start_position)

        # Check that the boundaries are in the correct order
        if line_end != -1 and line_end < end_position:
            extracted_text = content[line_end + 1:end_position].strip()

            extracted_books[book.name] = extracted_text
        else:
            print("Invalid boundaries:", book.name)
    else:
        print("Missing markers:", book.name)

print("Books successfully extracted:", len(extracted_books))
print("Total files:", len(book_files))


Books successfully extracted: 20
Total files: 20


### Validating the Extracted Dataset

Following extraction, the dataset will be checked to confirm that all 20 files have been processed successfully.

The validation will examine the number of extracted books, the length of each extracted text and whether any files contain empty results. This provides an initial check that the automated cleaning process has produced usable text for subsequent analysis.

Although these checks cannot guarantee that every text contains only narrative content, they can identify obvious extraction failures before the Exploratory Data Analysis stage.

In [12]:

# Display the character count for each extracted book
for filename, book_text in extracted_books.items():
    print(filename, "- Characters extracted:", len(book_text))

# Check for empty extracted texts
empty_books = [
    filename
    for filename, book_text in extracted_books.items()
    if not book_text
]

print("\nVALIDATION SUMMARY")
print("-" * 40)
print("Total source files:", len(book_files))
print("Books extracted:", len(extracted_books))
print("Empty extracted texts:", len(empty_books))

if len(extracted_books) == len(book_files) and not empty_books:
    print("Validation passed: all files contain extracted text.")
else:
    print("Validation requires further investigation.")
    print("Empty files:", empty_books)


1250-0.txt - Characters extracted: 108043
29019-0.txt - Characters extracted: 16622
84-0.txt - Characters extracted: 419335
pg16921.txt - Characters extracted: 336739
pg18247.txt - Characters extracted: 999981
pg18458.txt - Characters extracted: 348286
pg18846.txt - Characters extracted: 135647
pg19029.txt - Characters extracted: 33637
pg19090.txt - Characters extracted: 189734
pg19145.txt - Characters extracted: 352871
pg19651.txt - Characters extracted: 319551
pg19726.txt - Characters extracted: 237959
pg20788.txt - Characters extracted: 351073
pg20796.txt - Characters extracted: 260997
pg22544.txt - Characters extracted: 47423
pg23301.txt - Characters extracted: 35910
pg25550.txt - Characters extracted: 320747
pg29809.txt - Characters extracted: 456867
pg30427.txt - Characters extracted: 122157
pg32664.txt - Characters extracted: 131966

VALIDATION SUMMARY
----------------------------------------
Total source files: 20
Books extracted: 20
Empty extracted texts: 0
Validation passed: 

**Final Data Cleaning Validation**

The automated extraction process successfully processed all 20 Project Gutenberg text files. Validation confirmed that each file produced a non-empty extracted text, with no missing or empty results.

The cleaning process accounted for three variations in Gutenberg start and end markers, allowing the surrounding administrative headers and footers to be removed consistently across the dataset. The original source files were preserved, and the extracted texts were stored in a Python dictionary for subsequent analysis.

Although some books may retain additional non-literary material, such as tables of contents and publication information, this limitation has been documented. The dataset is now suitable for initial exploratory analysis, with further consideration of these limitations when interpreting stylistic measurements.

## Exploratory Data Analysis
Programmatically explore the dataset. Tell the reader, how the dataset looks like and at least three most interesting observations that you can learn from the data.

## Descriptive Analytics
Interogate the dataset with descriptive analytics. Answer 3 questions that you find interesting, and that can be answered using descriptive analytics. Make each question a subsection of this section and follow the steps: 1. the question, 2. justify why it is interesting 3. the answer.

## Suggestions
Suggest further steps to enrich the dataset with additional (external) data, or apply some additional advanced modelling or analysis. You do not have to fully implement your suggestions, but you will have to explain and justify your suggestions.

## Bibliography
Python Software Foundation (n.d.) *Built-in Types — str.find()*. Python Documentation. Available at: https://docs.python.org/3/library/stdtypes.html#str.find (Accessed: 10 November 2026).

Python Software Foundation (n.d.) Built-in Types — String Methods. Python Documentation. Available at: https://docs.python.org/3/library/stdtypes.html#string-methods (Accessed: 10 November 2026).